# ECG Training Notebook — DAT255

This notebook is responsible for training the CNN and BiLSTM models, tuning hyperparameters,
and saving all artifacts needed for the comparison in `comparison.ipynb`.


**Pipeline:**
1. Setup & reproducibility
2. Load dataset + build multi-hot labels
3. Train/val/test split (PTB-XL folds), normalisation, class weights
4. Data augmentation (time shift, Gaussian noise, amplitude scaling)
5. Model definitions
6. Focal loss
7. Keras Tuner hyperparameter search on CNN and BiLSTM
8. Final training runs, with and without augmentation
9. Save everything the comparison notebook will need

**Artifacts produced** (in `artifacts/`):
- `ecg_cnn_aug_final.keras`, `ecg_cnn_noaug_final.keras`
- `ecg_lstm_aug_final.keras`, `ecg_lstm_noaug_final.keras`
- `normalisation_params.npz`
- `eval_sets.npz` (X_val, y_val, X_test, y_test | for comparison notebook)
- `histories.pkl` (training curves for all runs)
- `tuner_results.json` (CNN tuner summary)
- `tuner_results_lstm.json` (BiLSTM tuner summary)

## 1. Setup

In [1]:
# Colab setup — run once
# !pip install -q wfdb keras tensorflow mlflow keras-tuner

In [2]:
import os, json, pickle, random, zipfile, ast
os.environ.pop("MLFLOW_TRACKING_URI", None)
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import wfdb
import keras
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
import keras_tuner as kt

from keras import layers, callbacks, optimizers
from sklearn.preprocessing import MultiLabelBinarizer
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

# Artifact directory
ART_DIR = "artifacts"
os.makedirs(ART_DIR, exist_ok=True)

tf.autograph.set_verbosity(0)
warnings.filterwarnings("ignore")

print(f"Keras   : {keras.__version__}")
print(f"TF      : {tf.__version__}")
print(f"Tuner   : {kt.__version__}")
print(f"Backend : {keras.backend.backend()}")

Keras   : 3.13.2
TF      : 2.21.0
Tuner   : 1.4.8
Backend : tensorflow


## 2. Load dataset & build labels

In [3]:
# Paths
ZIP_PATH = "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip"
DATA_DIR = "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"

if not os.path.isdir(DATA_DIR):
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(".")
    print("Done.")

# Load metadata
df = pd.read_csv(os.path.join(DATA_DIR, "ptbxl_database.csv"), index_col="ecg_id")
df["scp_codes"] = df["scp_codes"].apply(ast.literal_eval)

# Mapping superclasses to SCP
scp_df = pd.read_csv(os.path.join(DATA_DIR, "scp_statements.csv"), index_col=0)
scp_df = scp_df[scp_df["diagnostic"] == 1]
code_to_superclass = scp_df["diagnostic_class"].to_dict()

SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
NUM_CLASSES = len(SUPERCLASSES)

def extract_superclasses(scp_dict):
    return list({code_to_superclass[c] for c in scp_dict
                 if c in code_to_superclass})

df["superclasses"] = df["scp_codes"].apply(extract_superclasses)
df = df[df["superclasses"].map(len) > 0].copy()   # drop unlabelled
df = df[df["age"] <= 120].copy()                  # drop age outliers

mlb = MultiLabelBinarizer(classes=SUPERCLASSES)
labels = mlb.fit_transform(df["superclasses"])
df[SUPERCLASSES] = labels

print(f"Samples after filtering: {len(df)}")
print("Class distribution:")
for i, sc in enumerate(SUPERCLASSES):
    n = labels[:, i].sum()
    print(f"  {sc:5s}: {n:5d}  ({n/len(df)*100:5.1f}%)")

Samples after filtering: 21103
Class distribution:
  NORM :  9483  ( 44.9%)
  MI   :  5348  ( 25.3%)
  STTC :  5099  ( 24.2%)
  CD   :  4756  ( 22.5%)
  HYP  :  2593  ( 12.3%)


In [4]:
# Load signals (100 Hz, 1000 samples, 12 leads)

SAMPLING_RATE = 100
LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF",
              "V1", "V2", "V3", "V4", "V5", "V6"]

def _read_one(path):
    return wfdb.rdrecord(path).p_signal.astype(np.float32)

def load_raw_signals(df, data_dir, cache_path=None, n_workers=8):
    if cache_path and os.path.exists(cache_path):
        print(f"Loading cached signals from {cache_path}")
        return np.load(cache_path)

    col = "filename_lr"  # 100 Hz version
    paths = [os.path.join(data_dir, p) for p in df[col].values]

    N = len(paths)
    out = np.empty((N, 1000, 12), dtype=np.float32)

    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        for i, sig in enumerate(tqdm(pool.map(_read_one, paths),
                                     total=N, desc="Reading WFDB")):
            out[i] = sig

    if cache_path:
        print(f"Caching to {cache_path}")
        np.save(cache_path, out)

    return out

print("Loading ECG signals...")
cache_path = os.path.join(ART_DIR, "ptbxl_signals_100hz.npy")
X_all = load_raw_signals(df, DATA_DIR, cache_path=cache_path)
y_all = labels.astype(np.float32)
print(f"X_all: {X_all.shape}   y_all: {y_all.shape}")

Loading ECG signals...
Loading cached signals from artifacts/ptbxl_signals_100hz.npy
X_all: (21103, 1000, 12)   y_all: (21103, 5)


## 3. Split, normalise, class weights

In [5]:
# PTB-XL official split: folds 1-8 train, 9 val, 10 test
folds = df["strat_fold"].values
train_mask = folds <= 8
val_mask   = folds == 9
test_mask  = folds == 10

X_train_raw, y_train = X_all[train_mask], y_all[train_mask]
X_val_raw,   y_val   = X_all[val_mask],   y_all[val_mask]
X_test_raw,  y_test  = X_all[test_mask],  y_all[test_mask]

# Per-channel z-score (fit on train only)
train_mean = X_train_raw.mean(axis=(0, 1), keepdims=True)
train_std  = X_train_raw.std(axis=(0, 1), keepdims=True)
train_std[train_std == 0] = 1.0

def normalise(X):
    return np.clip((X - train_mean) / train_std, -10.0, 10.0)

X_train = normalise(X_train_raw)
X_val   = normalise(X_val_raw)
X_test  = normalise(X_test_raw)

# Save normalisation + test set for the comparison notebook
np.savez(os.path.join(ART_DIR, "normalisation_params.npz"),
         mean=train_mean, std=train_std)
np.savez(os.path.join(ART_DIR, "eval_sets.npz"),
         X_val=X_val, y_val=y_val,
         X_test=X_test, y_test=y_test)

print(f"Train: {X_train.shape}   Val: {X_val.shape}   Test: {X_test.shape}")

Train: (16872, 1000, 12)   Val: (2105, 1000, 12)   Test: (2126, 1000, 12)


In [6]:
# Class weights for multi-label BCE (neg/pos ratio per class)
pos = y_train.sum(axis=0)
neg = len(y_train) - pos
class_weights = neg / (pos + 1e-7)

print("Class weights (neg/pos ratio):")
for sc, w in zip(SUPERCLASSES, class_weights):
    print(f"  {sc:5s}: {w:.2f}")

Class weights (neg/pos ratio):
  NORM : 1.23
  MI   : 2.93
  STTC : 3.13
  CD   : 3.43
  HYP  : 7.12


## 4. Data augmentation

Three cheap, signal-preserving augmentations applied on the fly during training:
- **Random time shift** | rolls the signal by ±50 samples (±0.5s)
- **Gaussian noise** | σ=0.02 on normalised signal
- **Amplitude scaling** | random factor in [0.9, 1.1]

These simulate real-world variation (slightly different recording start times,
sensor noise, patient-to-patient amplitude differences).
All are applied with 50% probability per-sample.

In [7]:
def augment_ecg(x, y):
    # Random time shift (±50 samples)
    if tf.random.uniform(()) < 0.5:
        shift = tf.random.uniform((), -50, 51, dtype=tf.int32)
        x = tf.roll(x, shift, axis=0)

    # Gaussian noise
    if tf.random.uniform(()) < 0.5:
        x = x + tf.random.normal(tf.shape(x), mean=0.0, stddev=0.02)

    # Amplitude scaling
    if tf.random.uniform(()) < 0.5:
        scale = tf.random.uniform((), 0.9, 1.1)
        x = x * scale

    return x, y


def make_dataset(X, y, batch_size=64, augment=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(X), 2048), seed=SEED)
    if augment:
        ds = ds.map(augment_ecg, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


BATCH_SIZE = 64
train_ds = make_dataset(X_train, y_train, BATCH_SIZE, augment=True, shuffle=True)
val_ds   = make_dataset(X_val, y_val, BATCH_SIZE)
test_ds  = make_dataset(X_test, y_test, BATCH_SIZE)

## 5. Model definitions

- **1D-CNN** — 4 conv blocks (wide-to-narrow kernels), GAP, dense head
- **BiLSTM** — conv front-end + 2× bidirectional LSTM

In [8]:
INPUT_SHAPE = (1000, 12)

def _last_conv_name(model):
    return next(l.name for l in reversed(model.layers)
                if isinstance(l, layers.Conv1D))


def conv_block(x, filters, kernel_size, spatial_dropout=0.0, pool=True):
    x = layers.Conv1D(filters, kernel_size, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    if pool:
        x = layers.MaxPooling1D(pool_size=2)(x)
    if spatial_dropout > 0:
        x = layers.SpatialDropout1D(spatial_dropout)(x)
    return x


def build_cnn(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES,
              filters=(32, 64, 128, 256), dropout=0.3, dense_units=128):
    inputs = keras.Input(shape=input_shape, name="ecg_input")
    x = inputs
    kernels = [15, 7, 5, 3]
    for i, f in enumerate(filters):
        k = kernels[i] if i < len(kernels) else 3
        sd = 0.1 if i >= 2 else 0.0  # only the deeper blocks get spatial dropout
        x = conv_block(x, f, k, spatial_dropout=sd)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="sigmoid", name="predictions")(x)
    return keras.Model(inputs, outputs, name="cnn_1d")

def build_lstm(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES,
               dropout=0.3, dense_units=128,
               lstm_units_1=128, lstm_units_2=64):
    inputs = keras.Input(shape=input_shape, name="ecg_input")
    x = conv_block(inputs, 64, 15, spatial_dropout=0.0)   # 1000 -> 500
    x = conv_block(x,     128,  7, spatial_dropout=0.0)   #  500 -> 250
    x = conv_block(x,     128,  5, spatial_dropout=0.1)   #  250 -> 125
    x = layers.MaxPooling1D(2)(x)                         #  125 ->  62

    x = layers.Bidirectional(
        layers.LSTM(lstm_units_1, return_sequences=True, dropout=dropout))(x)
    x = layers.Bidirectional(
        layers.LSTM(lstm_units_2, dropout=dropout))(x)

    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="sigmoid", name="predictions")(x)
    return keras.Model(inputs, outputs, name="bilstm")

# Quick summary
for build_fn in [build_cnn, build_lstm]:
    m = build_fn()
    print(f"{m.name:12s}  params={m.count_params():>10,}  "
          f"gradcam_layer={_last_conv_name(m)}")
    del m


cnn_1d        params=   194,821  gradcam_layer=conv1d_3
bilstm        params=   596,741  gradcam_layer=conv1d_6


## 6. Focal Loss

Binary cross-entropy treats every sample equally. Focal loss down-weights
easy examples (where the model is already confident) and focuses gradient
on hard examples

$\text{FL}(p_t) = -\alpha (1-p_t)^\gamma \log(p_t)$

We use α=0.25, γ=2.0

In [9]:
def binary_focal_loss(alpha=0.25, gamma=2.0):
    def loss_fn(y_true, y_pred):
        eps = keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1.0 - y_pred)
        alpha_t = tf.where(tf.equal(y_true, 1), alpha, 1.0 - alpha)
        return -tf.reduce_mean(alpha_t * tf.pow(1.0 - pt, gamma) * tf.math.log(pt))
    loss_fn.__name__ = "binary_focal_loss"
    return loss_fn

## 7. Keras Tuner

Both architectures are tuned (separate searches to keep configs independent).
Hyperband is used because it kills bad trials early.
`max_epochs=20` per trial, `factor=3`.

### CNN search space

| Hyperparam      | Range                                        |
|-----------------|----------------------------------------------|
| filters\_mult   | [0.5, 1.0, 1.5, 2.0]   → scales (32, 64, 128, 256) |
| dropout         | [0.2, 0.3, 0.4, 0.5]                         |
| dense\_units    | [64, 128, 256]                              |
| learning\_rate  | log-uniform [1e-4, 3e-3]                    |
| loss            | {BCE, focal}                                 |

### BiLSTM search space

| Hyperparam      | Range                                        |
|-----------------|----------------------------------------------|
| lstm\_units\_1  | [64, 128, 256]                              |
| lstm\_units\_2  | [32, 64, 128]                               |
| dropout         | [0.2, 0.3, 0.4, 0.5]                         |
| dense\_units    | [64, 128, 256]                              |
| learning\_rate  | log-uniform [1e-4, 3e-3]                    |
| loss            | {BCE, focal}                                 |


In [ ]:
TUNE = False # Set to True to re-run Keras Tuner; defaults below are the best values already found
TUNER_MAX_EPOCHS = 20
TUNER_DIR = "keras_tuner"

In [10]:
# Define CNN tuner
def build_tunable_cnn(hp):
    mult = hp.Choice("filters_mult", [0.5, 1.0, 1.5, 2.0])
    dropout = hp.Float("dropout", 0.2, 0.5, step=0.1)
    dense_units = hp.Choice("dense_units", [64, 128, 256])
    lr = hp.Float("lr", 1e-4, 3e-3, sampling="log")
    loss_type = hp.Choice("loss", ["bce", "focal"])

    base_filters = (32, 64, 128, 256)
    filters = tuple(max(8, int(f * mult)) for f in base_filters)

    model = build_cnn(filters=filters, dropout=dropout, dense_units=dense_units)
    loss = binary_focal_loss() if loss_type == "focal" else "binary_crossentropy"
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss=loss,
        metrics=[keras.metrics.AUC(name="auc", multi_label=True)],
    )
    return model


if TUNE:
    tuner_cnn = kt.Hyperband(
        build_tunable_cnn,
        objective=kt.Objective("val_auc", direction="max"),
        max_epochs=TUNER_MAX_EPOCHS,
        factor=3,
        directory=TUNER_DIR,
        project_name="ecg_cnn",
        overwrite=False,
        seed=SEED,
    )
    tuner_cnn.search_space_summary()


In [11]:
# Run CNN tuner
if TUNE:
    tuner_cnn.search(
        train_ds,
        validation_data=val_ds,
        epochs=TUNER_MAX_EPOCHS,
        callbacks=[callbacks.EarlyStopping(monitor="val_auc", mode="max",
                                           patience=4, restore_best_weights=True)],
        verbose=1,
    )
    best_hps_cnn = tuner_cnn.get_best_hyperparameters(num_trials=1)[0]
    best_config_cnn = {k: best_hps_cnn.get(k) for k in
                       ["filters_mult", "dropout", "dense_units", "lr", "loss"]}
    print("Best CNN hyperparameters:", best_config_cnn)

    with open(os.path.join(ART_DIR, "tuner_results.json"), "w") as f:
        json.dump(best_config_cnn, f, indent=2)
else:
    # Best values from a previous Hyperband run (val_auc = 0.9272)
    best_config_cnn = {"filters_mult": 2.0, "dropout": 0.3,
                       "dense_units": 128, "lr": 4.098521113058352e-4,
                       "loss": "bce"}
    with open(os.path.join(ART_DIR, "tuner_results.json"), "w") as f:
        json.dump(best_config_cnn, f, indent=2)
    print("CNN tuning skipped. Using defaults:", best_config_cnn)


CNN tuning skipped. Using defaults: {'filters_mult': 2.0, 'dropout': 0.3, 'dense_units': 128, 'lr': 0.0004098521113058352, 'loss': 'bce'}


In [12]:
# Define BiLSTM tuner
def build_tunable_lstm(hp):
    lstm_units_1 = hp.Choice("lstm_units_1", [64, 128, 256])
    lstm_units_2 = hp.Choice("lstm_units_2", [32, 64, 128])
    dropout = hp.Float("dropout", 0.2, 0.5, step=0.1)
    dense_units = hp.Choice("dense_units", [64, 128, 256])
    lr = hp.Float("lr", 1e-4, 3e-3, sampling="log")
    loss_type = hp.Choice("loss", ["bce", "focal"])

    model = build_lstm(dropout=dropout, dense_units=dense_units,
                       lstm_units_1=lstm_units_1, lstm_units_2=lstm_units_2)
    loss = binary_focal_loss() if loss_type == "focal" else "binary_crossentropy"
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss=loss,
        metrics=[keras.metrics.AUC(name="auc", multi_label=True)],
    )
    return model


if TUNE:
    tuner_lstm = kt.Hyperband(
        build_tunable_lstm,
        objective=kt.Objective("val_auc", direction="max"),
        max_epochs=TUNER_MAX_EPOCHS,
        factor=3,
        directory=TUNER_DIR,
        project_name="ecg_lstm",
        overwrite=False,
        seed=SEED,
    )
    tuner_lstm.search_space_summary()


In [13]:
# Run BiLSTM tuner
if TUNE:
    tuner_lstm.search(
        train_ds,
        validation_data=val_ds,
        epochs=TUNER_MAX_EPOCHS,
        callbacks=[callbacks.EarlyStopping(monitor="val_auc", mode="max",
                                           patience=4, restore_best_weights=True)],
        verbose=1,
    )
    best_hps_lstm = tuner_lstm.get_best_hyperparameters(num_trials=1)[0]
    best_config_lstm = {k: best_hps_lstm.get(k) for k in
                        ["lstm_units_1", "lstm_units_2", "dropout",
                         "dense_units", "lr", "loss"]}
    print("Best BiLSTM hyperparameters:", best_config_lstm)

    with open(os.path.join(ART_DIR, "tuner_results_lstm.json"), "w") as f:
        json.dump(best_config_lstm, f, indent=2)
else:
    # Best values from a previous Hyperband run (val_auc = 0.9249)
    best_config_lstm = {"lstm_units_1": 128, "lstm_units_2": 128, "dropout": 0.2,
                        "dense_units": 128, "lr": 3.9447639218573435e-4,
                        "loss": "bce"}
    with open(os.path.join(ART_DIR, "tuner_results_lstm.json"), "w") as f:
        json.dump(best_config_lstm, f, indent=2)
    print("BiLSTM tuning skipped. Using defaults:", best_config_lstm)


BiLSTM tuning skipped. Using defaults: {'lstm_units_1': 128, 'lstm_units_2': 128, 'dropout': 0.2, 'dense_units': 128, 'lr': 0.00039447639218573435, 'loss': 'bce'}


## 8. Final training runs

In [14]:
# MLflow setup and training setup
try:
    import mlflow
    mlflow.set_tracking_uri("file:./mlruns")
    mlflow.set_experiment("ECG_PTBXL_Final")
    USE_MLFLOW = True
except Exception as e:
    print(f"MLflow not available ({e}); continuing without tracking.")
    USE_MLFLOW = False


def train_model(model, name, loss_name="focal", lr=1e-3,
                epochs=50, use_augmentation=True):
    loss_fn = binary_focal_loss() if loss_name == "focal" else "binary_crossentropy"
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss=loss_fn,
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.AUC(name="auc", multi_label=True),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )

    suffix = "aug" if use_augmentation else "noaug"
    ckpt_path = os.path.join(ART_DIR, f"ecg_{name}_{suffix}_final.keras")
    cb = [
        callbacks.ModelCheckpoint(ckpt_path, monitor="val_auc", mode="max",
                                  save_best_only=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                    patience=5, min_lr=1e-6, verbose=1),
        callbacks.EarlyStopping(monitor="val_auc", mode="max",
                                patience=10, restore_best_weights=True, verbose=1),
    ]

    tr_ds = train_ds if use_augmentation else make_dataset(
        X_train, y_train, BATCH_SIZE, augment=False, shuffle=True)

    run_name = f"{name}_{suffix}"
    if USE_MLFLOW:
        with mlflow.start_run(run_name=run_name):
            mlflow.log_params({"loss": loss_name, "lr": lr, "epochs": epochs,
                               "augmentation": use_augmentation,
                               "n_params": model.count_params()})
            history = model.fit(tr_ds, validation_data=val_ds,
                                epochs=epochs, callbacks=cb, verbose=2)
            for k, v in history.history.items():
                for i, val in enumerate(v):
                    mlflow.log_metric(k, val, step=i)
    else:
        history = model.fit(tr_ds, validation_data=val_ds,
                            epochs=epochs, callbacks=cb, verbose=2)
    return history


In [17]:
# CNN with AUGMENTATION
mult = best_config_cnn["filters_mult"]
tuned_filters = tuple(max(8, int(f * mult)) for f in (32, 64, 128, 256))

cnn_model_aug = build_cnn(
    filters=tuned_filters,
    dropout=best_config_cnn["dropout"],
    dense_units=best_config_cnn["dense_units"],
)
cnn_aug_history = train_model(cnn_model_aug, "cnn",
                              loss_name=best_config_cnn["loss"],
                              lr=best_config_cnn["lr"],
                              epochs=50,
                              use_augmentation=True)


Epoch 1/50

Epoch 1: val_auc improved from None to 0.89700, saving model to artifacts/ecg_cnn_aug_final.keras

Epoch 1: finished saving model to artifacts/ecg_cnn_aug_final.keras
264/264 - 10s - 39ms/step - accuracy: 0.8438 - auc: 0.8628 - loss: 0.3613 - precision: 0.7430 - recall: 0.6056 - val_accuracy: 0.8453 - val_auc: 0.8970 - val_loss: 0.3509 - val_precision: 0.7664 - val_recall: 0.5762 - learning_rate: 4.0985e-04
Epoch 2/50

Epoch 2: val_auc improved from 0.89700 to 0.91182, saving model to artifacts/ecg_cnn_aug_final.keras

Epoch 2: finished saving model to artifacts/ecg_cnn_aug_final.keras
264/264 - 3s - 10ms/step - accuracy: 0.8688 - auc: 0.8987 - loss: 0.3127 - precision: 0.7840 - recall: 0.6802 - val_accuracy: 0.8618 - val_auc: 0.9118 - val_loss: 0.3153 - val_precision: 0.7686 - val_recall: 0.6642 - learning_rate: 4.0985e-04
Epoch 3/50

Epoch 3: val_auc did not improve from 0.91182
264/264 - 3s - 9ms/step - accuracy: 0.8772 - auc: 0.9111 - loss: 0.2939 - precision: 0.7961 - 

In [16]:
# CNN without AUGMENTATION
cnn_model_noaug = build_cnn(
    filters=tuned_filters,
    dropout=best_config_cnn["dropout"],
    dense_units=best_config_cnn["dense_units"],
)
cnn_noaug_history = train_model(cnn_model_noaug, "cnn",
                                loss_name=best_config_cnn["loss"],
                                lr=best_config_cnn["lr"],
                                epochs=50,
                                use_augmentation=False)


Epoch 1/50

Epoch 1: val_auc improved from None to 0.90071, saving model to artifacts/ecg_cnn_noaug_final.keras

Epoch 1: finished saving model to artifacts/ecg_cnn_noaug_final.keras
264/264 - 26s - 98ms/step - accuracy: 0.8468 - auc: 0.8676 - loss: 0.3547 - precision: 0.7508 - recall: 0.6108 - val_accuracy: 0.8496 - val_auc: 0.9007 - val_loss: 0.3503 - val_precision: 0.7535 - val_recall: 0.6200 - learning_rate: 4.0985e-04
Epoch 2/50

Epoch 2: val_auc improved from 0.90071 to 0.91097, saving model to artifacts/ecg_cnn_noaug_final.keras

Epoch 2: finished saving model to artifacts/ecg_cnn_noaug_final.keras
264/264 - 3s - 13ms/step - accuracy: 0.8727 - auc: 0.9052 - loss: 0.3045 - precision: 0.7927 - recall: 0.6876 - val_accuracy: 0.8604 - val_auc: 0.9110 - val_loss: 0.3160 - val_precision: 0.7675 - val_recall: 0.6587 - learning_rate: 4.0985e-04
Epoch 3/50

Epoch 3: val_auc improved from 0.91097 to 0.91501, saving model to artifacts/ecg_cnn_noaug_final.keras

Epoch 3: finished saving mod

In [18]:
# BiLSTM with AUGMENTATION
lstm_model_aug = build_lstm(
    dropout=best_config_lstm["dropout"],
    dense_units=best_config_lstm["dense_units"],
    lstm_units_1=best_config_lstm["lstm_units_1"],
    lstm_units_2=best_config_lstm["lstm_units_2"],
)
lstm_aug_history = train_model(lstm_model_aug, "lstm",
                               loss_name=best_config_lstm["loss"],
                               lr=best_config_lstm["lr"],
                               epochs=40,
                               use_augmentation=True)


Epoch 1/40

Epoch 1: val_auc improved from None to 0.87528, saving model to artifacts/ecg_lstm_aug_final.keras

Epoch 1: finished saving model to artifacts/ecg_lstm_aug_final.keras
264/264 - 16s - 61ms/step - accuracy: 0.8218 - auc: 0.8255 - loss: 0.3991 - precision: 0.7056 - recall: 0.5337 - val_accuracy: 0.8308 - val_auc: 0.8753 - val_loss: 0.3974 - val_precision: 0.7173 - val_recall: 0.5681 - learning_rate: 3.9448e-04
Epoch 2/40

Epoch 2: val_auc improved from 0.87528 to 0.89960, saving model to artifacts/ecg_lstm_aug_final.keras

Epoch 2: finished saving model to artifacts/ecg_lstm_aug_final.keras
264/264 - 12s - 47ms/step - accuracy: 0.8606 - auc: 0.8834 - loss: 0.3312 - precision: 0.7718 - recall: 0.6550 - val_accuracy: 0.8591 - val_auc: 0.8996 - val_loss: 0.3262 - val_precision: 0.7656 - val_recall: 0.6543 - learning_rate: 3.9448e-04
Epoch 3/40

Epoch 3: val_auc did not improve from 0.89960
264/264 - 12s - 44ms/step - accuracy: 0.8685 - auc: 0.8979 - loss: 0.3122 - precision: 0.

In [19]:
# BiLSTM without AUGMENTATION
lstm_model_noaug = build_lstm(
    dropout=best_config_lstm["dropout"],
    dense_units=best_config_lstm["dense_units"],
    lstm_units_1=best_config_lstm["lstm_units_1"],
    lstm_units_2=best_config_lstm["lstm_units_2"],
)
lstm_noaug_history = train_model(lstm_model_noaug, "lstm",
                                 loss_name=best_config_lstm["loss"],
                                 lr=best_config_lstm["lr"],
                                 epochs=40,
                                 use_augmentation=False)


Epoch 1/40

Epoch 1: val_auc improved from None to 0.88126, saving model to artifacts/ecg_lstm_noaug_final.keras

Epoch 1: finished saving model to artifacts/ecg_lstm_noaug_final.keras
264/264 - 13s - 50ms/step - accuracy: 0.8240 - auc: 0.8267 - loss: 0.3973 - precision: 0.7114 - recall: 0.5382 - val_accuracy: 0.8503 - val_auc: 0.8813 - val_loss: 0.3482 - val_precision: 0.7600 - val_recall: 0.6134 - learning_rate: 3.9448e-04
Epoch 2/40

Epoch 2: val_auc improved from 0.88126 to 0.89846, saving model to artifacts/ecg_lstm_noaug_final.keras

Epoch 2: finished saving model to artifacts/ecg_lstm_noaug_final.keras
264/264 - 10s - 37ms/step - accuracy: 0.8614 - auc: 0.8853 - loss: 0.3291 - precision: 0.7728 - recall: 0.6576 - val_accuracy: 0.8637 - val_auc: 0.8985 - val_loss: 0.3257 - val_precision: 0.7843 - val_recall: 0.6506 - learning_rate: 3.9448e-04
Epoch 3/40

Epoch 3: val_auc improved from 0.89846 to 0.89972, saving model to artifacts/ecg_lstm_noaug_final.keras

Epoch 3: finished savi

## 9. Save histories for the comparison notebook

In [20]:
histories = {
    "cnn_aug":     cnn_aug_history.history,
    "cnn_noaug":   cnn_noaug_history.history,
    "lstm_aug":    lstm_aug_history.history,
    "lstm_noaug":  lstm_noaug_history.history,
}
with open(os.path.join(ART_DIR, "histories.pkl"), "wb") as f:
    pickle.dump(histories, f)

print("Training complete. Artifacts saved:")
for f in sorted(os.listdir(ART_DIR)):
    size_mb = os.path.getsize(os.path.join(ART_DIR, f)) / 1e6
    print(f"  {f:40s} {size_mb:>8.2f} MB")


Training complete. Artifacts saved:
  ecg_cnn_aug_final.keras                      8.43 MB
  ecg_cnn_final.keras                          2.42 MB
  ecg_cnn_noaug_final.keras                    8.43 MB
  ecg_lstm_aug_final.keras                    10.23 MB
  ecg_lstm_final.keras                         7.27 MB
  ecg_lstm_noaug_final.keras                  10.23 MB
  ecg_rescnn_final.keras                      19.54 MB
  ecg_xresnet1d_final.keras                  222.99 MB
  eval_sets.npz                              203.17 MB
  histories.pkl                                0.01 MB
  normalisation_params.npz                     0.00 MB
  ptbxl_signals_100hz.npy                   1012.94 MB
  ptbxl_signals_100hz_21103.npy             1012.94 MB
  thresholds.json                              0.00 MB
  tuner_results.json                           0.00 MB
  tuner_results_lstm.json                      0.00 MB


In [24]:
keras.utils.plot_model(cnn_model_aug, show_shapes=True, show_layer_names=True)
print(f"CNN_AUG:")
keras.utils.plot_model(cnn_model_noaug, show_shapes=True, show_layer_names=True)
print(f"CNN_NOAUG:")
keras.utils.plot_model(lstm_model_aug, show_shapes=True, show_layer_names=True)
print(f"LSTM_AUG:")
keras.utils.plot_model(lstm_model_noaug, show_shapes=True, show_layer_names=True)
print(f"LSTM_NOAUG:")

You must install pydot (`pip install pydot`) for `plot_model` to work.
CNN_AUG:
You must install pydot (`pip install pydot`) for `plot_model` to work.
CNN_NOAUG:
You must install pydot (`pip install pydot`) for `plot_model` to work.
LSTM_AUG:
You must install pydot (`pip install pydot`) for `plot_model` to work.
LSTM_NOAUG:
